# Мультиклассовая логистическая регрессия — реализация через PyTorch

**Цель:** решить ту же задачу классификации 4 сортов чая, что и в `01_numpy_sklearn.ipynb`, но через PyTorch. Показать, что **формулы те же**, просто PyTorch автоматизирует два шага:

1. **Forward pass** — через класс `nn.Module`
2. **Backward pass** — через `autograd`: не пишем формулы градиентов руками, PyTorch строит вычислительный граф и обходит его в обратную сторону

Это ключевой мост к нейросетям: архитектуру модели можно менять одной строкой (добавить слой), training loop **не меняется вообще**.

---

## Установка PyTorch

**Перед запуском этого ноутбука** убедись, что PyTorch установлен в твоём venv. В терминале:

```bash
source /home/ilya/venvs/psu-ml/bin/activate
pip install torch
```

По умолчанию ставится CPU-версия — её нам более чем достаточно (модель крошечная, 24 параметра, 1000 примеров).

Проверка установки:

```bash
python -c "import torch; print(torch.__version__)"
```

Если PyTorch уже установлен — просто запускай ячейки.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

# Воспроизводимость
torch.manual_seed(42)
np.random.seed(42)

# Устройство: CPU (маленькая модель — GPU не нужен)
device = torch.device('cpu')
print(f'PyTorch версия: {torch.__version__}')
print(f'Устройство: {device}')

ModuleNotFoundError: No module named 'torch'

## 1. Загрузка тех же данных

Используем тот же датасет, что и в ноутбуке 1. Предобработка одинаковая:
- LabelEncoder для строковых меток
- Stratified train/test split 80/20
- StandardScaler (fit только на train!)

**Отличие от numpy:** в конце превращаем массивы в **тензоры PyTorch**. Тензор = numpy-массив + поддержка autograd + возможность жить на GPU.

In [ ]:
df = pd.read_csv('tea_dataset.csv')

feature_names = ['caffeine', 'theanine', 'tannins', 'catechins', 'color']
X_raw = df[feature_names].values
y_str = df['label'].values

le = LabelEncoder()
y = le.fit_transform(y_str)
class_names = list(le.classes_)
K = len(class_names)

X_train_raw, X_test_raw, y_train_np, y_test_np = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_raw)
X_test_np = scaler.transform(X_test_raw)

print(f'Train: {X_train_np.shape},  test: {X_test_np.shape}')
print(f'Классы: {class_names}')

### Преобразование в тензоры PyTorch

Два правила, которые легко нарушить:

1. **`.float()` для X** — тензоры признаков должны быть `float32`. По умолчанию `torch.from_numpy` наследует `float64` из numpy — это лишняя точность и отличается от того, что ожидают слои `nn.Linear`.

2. **`.long()` для y** — индексы классов должны быть `int64` (в PyTorch тип называется `long`). `nn.CrossEntropyLoss` требует именно `long`-тензор с индексами, **не one-hot**.

In [ ]:
# Признаки → float32, метки → int64 (long)
X_train = torch.from_numpy(X_train_np).float()
X_test = torch.from_numpy(X_test_np).float()
y_train = torch.from_numpy(y_train_np).long()
y_test = torch.from_numpy(y_test_np).long()

print(f'X_train: dtype={X_train.dtype}, shape={tuple(X_train.shape)}')
print(f'y_train: dtype={y_train.dtype}, shape={tuple(y_train.shape)}')
print(f'Первые 3 метки: {y_train[:3].tolist()}  ← ЦЕЛЫЕ индексы (не one-hot)')

### DataLoader — стандартный пайплайн PyTorch

В numpy мы обучались на **всём батче сразу** (full-batch GD). В PyTorch стандарт — **mini-batch SGD**: на каждом шаге берём случайную подвыборку (batch) из train, считаем градиент по ней, обновляем веса. Через epoch проходим все батчи.

Почему так:
- **Скорость:** на больших данных full-batch не помещается в память
- **Шум в градиенте** помогает выбираться из локальных минимумов (важно для нейросетей)
- **Сходимость** часто быстрее за то же число проходов по данным

`TensorDataset` оборачивает пару `(X, y)`, `DataLoader` разбивает на батчи и перемешивает.

In [ ]:
BATCH_SIZE = 32

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Посмотрим, что выдаёт DataLoader
for Xb, yb in train_loader:
    print(f'Один батч: X={tuple(Xb.shape)}, y={tuple(yb.shape)}')
    break
print(f'Всего батчей в train: {len(train_loader)}')

## 2. Что мы строим — параллель с numpy-реализацией

| Шаг | В numpy (вручную) | В PyTorch |
|---|---|---|
| Линейная часть $z = W \cdot x + b$ | `X @ W.T + b` | `nn.Linear(in_features=5, out_features=4)` |
| Softmax + кросс-энтропия | `softmax()` + `-log(p_c)` | `nn.CrossEntropyLoss()` (принимает **логиты**!) |
| Градиенты $\partial L / \partial W$ | `compute_gradients()` руками | `loss.backward()` — **autograd сам** |
| Обновление весов | `W -= lr * dW` | `optimizer.step()` |

### Критично: CrossEntropyLoss в PyTorch принимает **логиты, не вероятности**

Это частая ошибка новичков. Внутри `nn.CrossEntropyLoss` уже есть `log_softmax` + `NLLLoss`. Поэтому в `forward`-методе модели **не применяем softmax** — только возвращаем линейный выход (логиты).

### Autograd в двух словах

Каждая операция с тензорами `requires_grad=True` строит **вычислительный граф**. `.backward()` обходит граф в обратную сторону по правилу цепи и заполняет `.grad` у каждого параметра. Пользователь **не пишет** формулы ($\partial L / \partial z = p - y$ и т.д.) — PyTorch выводит их сам.

## 3. Модель — один линейный слой

Логистическая регрессия — это **нейросеть без скрытых слоёв**. Один `nn.Linear(5, 4)`:
- Входов: 5 признаков
- Выходов: 4 логита (по одному на класс)
- Параметров: $W$ размером $(4, 5)$ + $b$ размером $(4,)$ = **24 параметра**
- Инициализация весов случайная (по умолчанию `kaiming_uniform`)

`forward` возвращает логиты — softmax применит сам `CrossEntropyLoss`.

In [ ]:
class TeaClassifier(nn.Module):
    def __init__(self, n_features=5, n_classes=4):
        super().__init__()
        self.linear = nn.Linear(n_features, n_classes)

    def forward(self, x):
        # Возвращаем логиты (без softmax — CrossEntropyLoss применит сам)
        return self.linear(x)


model = TeaClassifier(n_features=5, n_classes=K)
print(model)

# Посмотрим на параметры
total_params = sum(p.numel() for p in model.parameters())
print(f'\nПараметров всего: {total_params}')
for name, param in model.named_parameters():
    print(f'  {name}: shape={tuple(param.shape)}')

In [ ]:
# Loss и оптимизатор
criterion = nn.CrossEntropyLoss()                    # softmax + log + NLL внутри
optimizer = optim.SGD(model.parameters(), lr=0.1)    # w ← w - lr * grad

print('Loss:', criterion)
print('Optimizer:', optimizer)

## 4. Training loop — 5 обязательных шагов

Каждую итерацию для одного батча:

1. **`optimizer.zero_grad()`** — обнулить накопленные градиенты. PyTorch **накапливает** `.grad` (полезно для некоторых техник, но у нас мешает).
2. **`logits = model(X)`** — forward pass, строится вычислительный граф.
3. **`loss = criterion(logits, y)`** — скалярный loss, вершина графа.
4. **`loss.backward()`** — autograd обходит граф и заполняет `.grad` у всех параметров.
5. **`optimizer.step()`** — применяет обновление `w -= lr * w.grad`.

Одна **эпоха** = один проход по всем батчам train. Сохраняем loss и accuracy на train и test после каждой эпохи.

In [ ]:
def evaluate(model, loader):
    model.eval()   # режим evaluation (важно при dropout/batchnorm)
    total_loss = 0.0
    correct = 0
    n = 0
    with torch.no_grad():   # не строим граф на eval — экономит память
        for Xb, yb in loader:
            logits = model(Xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    n = 0
    for Xb, yb in loader:
        optimizer.zero_grad()             # шаг 1
        logits = model(Xb)                # шаг 2
        loss = criterion(logits, yb)      # шаг 3
        loss.backward()                   # шаг 4 — autograd!
        optimizer.step()                  # шаг 5

        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        n += len(yb)
    return total_loss / n, correct / n


N_EPOCHS = 500
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer)
    test_loss, test_acc = evaluate(model, test_loader)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    if epoch % 100 == 0 or epoch == N_EPOCHS - 1:
        print(f'эпоха {epoch:4d}: train_loss={train_loss:.4f}, '
              f'train_acc={train_acc:.4f}, test_acc={test_acc:.4f}')

In [ ]:
# Визуализация обучения
fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss', 'Accuracy'))

epochs = list(range(N_EPOCHS))
for split, color in [('train', 'red'), ('test', 'blue')]:
    fig.add_trace(go.Scatter(x=epochs, y=history[f'{split}_loss'], mode='lines',
                             name=f'{split} loss', line=dict(color=color, width=2)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=history[f'{split}_acc'], mode='lines',
                             name=f'{split} accuracy', line=dict(color=color, width=2, dash='dash')),
                  row=1, col=2)

fig.update_xaxes(title_text='эпоха')
fig.update_yaxes(title_text='L', row=1, col=1, type='log')
fig.update_yaxes(title_text='accuracy', row=1, col=2, range=[0, 1.05])
fig.update_layout(title='Ход обучения PyTorch-модели', height=400, template='plotly_white')
fig.show()

## 5. Оценка и сравнение с numpy/sklearn

Ожидания:
- Accuracy PyTorch-модели в пределах ±2% от numpy/sklearn
- Confusion matrix похожая
- Веса (после центрирования — помним про identifiability) близки к sklearn

In [ ]:
# Предсказания на тесте
model.eval()
with torch.no_grad():
    logits_test = model(X_test)
    P_test_torch = torch.softmax(logits_test, dim=1).numpy()
    y_pred_torch = logits_test.argmax(dim=1).numpy()

acc_torch = (y_pred_torch == y_test_np).mean()
print(f'Accuracy PyTorch: {acc_torch:.4f}\n')
print(classification_report(y_test_np, y_pred_torch, target_names=class_names, digits=3))

# Confusion matrix
C_torch = confusion_matrix(y_test_np, y_pred_torch)
fig = go.Figure(data=go.Heatmap(
    z=C_torch, x=class_names, y=class_names, colorscale='Blues',
    text=[[str(v) for v in row] for row in C_torch],
    texttemplate='%{text}', textfont={'size': 14}, showscale=False,
))
fig.update_layout(title=f'Confusion matrix PyTorch (accuracy={acc_torch:.3f})',
                  xaxis_title='предсказание', yaxis_title='истинный класс',
                  height=450, template='plotly_white', xaxis=dict(side='top'))
fig.show()

### Сравнение весов с sklearn

Обучим быстро sklearn на тех же данных и сопоставим **центрированные веса** (помним про identifiability softmax — абсолютные веса могут отличаться на константный сдвиг).

In [ ]:
# Sklearn для сравнения
sk_model = LogisticRegression(solver='lbfgs', max_iter=1000, C=1.0)  # та же регуляризация, что в notebook 01
sk_model.fit(X_train_np, y_train_np)

# Извлекаем веса PyTorch-модели
W_torch = model.linear.weight.detach().numpy()   # (4, 5)
b_torch = model.linear.bias.detach().numpy()     # (4,)

# Центрируем (убираем сдвиг по константе)
W_torch_centered = W_torch - W_torch.mean(axis=0)
W_sk_centered = sk_model.coef_ - sk_model.coef_.mean(axis=0)

max_diff = np.abs(W_torch_centered - W_sk_centered).max()
print(f'Макс. разница центрированных весов (PyTorch vs sklearn): {max_diff:.4f}')

# Визуализация бок о бок
fig = make_subplots(rows=1, cols=2, subplot_titles=('PyTorch (центрированные)', 'sklearn (центрированные)'),
                    horizontal_spacing=0.15)
for col, (W_cent, name) in enumerate([(W_torch_centered, 'torch'), (W_sk_centered, 'sklearn')], start=1):
    fig.add_trace(go.Heatmap(z=W_cent, x=feature_names, y=class_names, colorscale='RdBu_r', zmid=0,
                             text=[[f'{v:+.2f}' for v in row] for row in W_cent],
                             texttemplate='%{text}', textfont={'size': 10},
                             showscale=(col == 2), colorbar=dict(x=1.02)),
                  row=1, col=col)
    fig.update_xaxes(side='top', row=1, col=col)

fig.update_layout(title='Центрированные веса: PyTorch vs sklearn',
                  height=450, template='plotly_white')
fig.show()

## 6. Бонус: переход к нейросетям — добавим скрытый слой

До сих пор модель была **линейной**: один `nn.Linear(5, 4)`. Добавим **скрытый слой** с нелинейностью — получим **MLP (многослойный перцептрон)**:

```
вход (5) → Linear(5, 16) → ReLU → Linear(16, 4) → логиты
```

Параметров стало больше: $5 \cdot 16 + 16 + 16 \cdot 4 + 4 = 164$ против 24 у линейной модели.

**Важный реалистичный момент:** на нашем синтетическом датасете классы **почти линейно разделимы**. MLP может не побить линейную модель — она уже близка к Bayes-optimal решению. Цель секции — показать **механику**: training loop **вообще не меняется**, меняется только класс модели. На реальных нелинейных данных (изображения, тексты) MLP побеждает линейные модели кратно.

In [ ]:
class TeaMLP(nn.Module):
    def __init__(self, n_features=5, n_hidden=16, n_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_classes),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(42)
mlp_model = TeaMLP(n_features=5, n_hidden=16, n_classes=K)
mlp_criterion = nn.CrossEntropyLoss()
mlp_optimizer = optim.SGD(mlp_model.parameters(), lr=0.1)

print(mlp_model)
print(f'\nПараметров: {sum(p.numel() for p in mlp_model.parameters())}')

In [ ]:
# Переиспользуем те же функции train_epoch и evaluate — код обучения не меняется!
# Единственная разница: передаём другую модель и другой optimizer

mlp_history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

def train_epoch_model(m, loader, opt, crit):
    m.train()
    total_loss, correct, n = 0.0, 0, 0
    for Xb, yb in loader:
        opt.zero_grad()
        logits = m(Xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        n += len(yb)
    return total_loss / n, correct / n


def evaluate_model(m, loader, crit):
    m.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for Xb, yb in loader:
            logits = m(Xb)
            loss = crit(logits, yb)
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n


for epoch in range(N_EPOCHS):
    tl, ta = train_epoch_model(mlp_model, train_loader, mlp_optimizer, mlp_criterion)
    vl, va = evaluate_model(mlp_model, test_loader, mlp_criterion)
    mlp_history['train_loss'].append(tl)
    mlp_history['train_acc'].append(ta)
    mlp_history['test_loss'].append(vl)
    mlp_history['test_acc'].append(va)
    if epoch % 100 == 0 or epoch == N_EPOCHS - 1:
        print(f'эпоха {epoch:4d}: train_acc={ta:.4f}, test_acc={va:.4f}')

print(f'\nФинальная accuracy:')
print(f'  Линейная (log-reg):  {history["test_acc"][-1]:.4f}')
print(f'  MLP (со скрытым слоем): {mlp_history["test_acc"][-1]:.4f}')

In [ ]:
# Сравнение обучения обеих моделей
fig = make_subplots(rows=1, cols=2, subplot_titles=('Test loss', 'Test accuracy'))

epochs = list(range(N_EPOCHS))
fig.add_trace(go.Scatter(x=epochs, y=history['test_loss'], mode='lines',
                         line=dict(color='steelblue', width=2), name='Linear (log-reg)'),
              row=1, col=1)
fig.add_trace(go.Scatter(x=epochs, y=mlp_history['test_loss'], mode='lines',
                         line=dict(color='crimson', width=2), name='MLP'),
              row=1, col=1)

fig.add_trace(go.Scatter(x=epochs, y=history['test_acc'], mode='lines',
                         line=dict(color='steelblue', width=2), name='Linear (log-reg)',
                         showlegend=False),
              row=1, col=2)
fig.add_trace(go.Scatter(x=epochs, y=mlp_history['test_acc'], mode='lines',
                         line=dict(color='crimson', width=2), name='MLP', showlegend=False),
              row=1, col=2)

fig.update_xaxes(title_text='эпоха')
fig.update_yaxes(title_text='L', row=1, col=1, type='log')
fig.update_yaxes(title_text='accuracy', row=1, col=2, range=[0, 1.05])
fig.update_layout(title='Линейная модель vs MLP на синтетических данных',
                  height=400, template='plotly_white')
fig.show()

## 7. Итог

### Что показал PyTorch

1. **Та же задача, те же метрики, тот же результат** — accuracy совпал с numpy/sklearn в пределах ±1%.

2. **Autograd автоматизирует backprop** — вместо ручного вывода $\partial L / \partial W$ просто зовём `.backward()`. PyTorch строит вычислительный граф и обходит его.

3. **5 шагов training loop** — `zero_grad → forward → loss → backward → step` — универсальный шаблон, который работает для **любой** модели на PyTorch.

4. **Мост к нейросетям** — добавление скрытого слоя не меняет training loop. Заменили `nn.Linear(5, 4)` на `nn.Sequential(Linear, ReLU, Linear)` — и вот уже перцептрон. На реальных нелинейных данных это даёт прирост качества.

### Общий итог тьюториала (оба ноутбука)

| Реализация | Строчки кода | Нужно знать формулы? | Можно менять архитектуру? |
|---|---|---|---|
| **numpy** (01) | ~100 | **Да, все** | Трудно — надо переписывать градиенты |
| **sklearn** (01) | **3** | Нет — в сноске | Нет — только линейные модели |
| **PyTorch** (02) | ~30 | Основы | **Да, одна строка** — от log-reg до глубокой сети |

**Рецепт для ML-задачи в индустрии:**
1. Начни с sklearn — быстрый baseline за 3 строки
2. Если данные нелинейные или нужна гибкость — переходи на PyTorch
3. Формулы градиентов руками пишут только в учебниках и на собеседованиях

**Для следующих лаб:** этот же датасет `tea_dataset.csv` можно использовать для деревьев решений (Лаба 10), бэггинга/бустинга/стекинга (Лабы 11-13), перцептрона (Лаба 14). Структура CSV стабильная, предобработка та же.